# M5 L5: Document Ingestion and Parsing

**Build an idempotent ingestion service with content-hash dedupe and a DLQ.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

**Ingestion** is the front door of every search system. Documents come in as text files, web pages, Word docs — and you need to clean them up, store them somewhere, and never accidentally store the same document twice.

The two non-negotiable rules: **(1) running the pipeline twice gives the same result** (idempotency), and **(2) one bad document never stops the whole batch** (dead-letter queue). Get those two right and most production failures disappear.

## The 5 things to know

1. **Parse, normalize, dedupe, persist** — the four sub-problems, in that order.
2. **Content-hash dedupe** — SHA-256 of the normalized text; same hash = same doc.
3. **Idempotency** — re-running is safe; same input = same store state.
4. **Dead-letter queue** — bad docs go to a side channel, never silently dropped.
5. **Version your parsers** — silent parse changes corrupt the index; log every parse.

In [1]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm05-ingestion' / 'L5-document-ingestion.py'
_spec = importlib.util.spec_from_file_location('L5', _p)
_L5 = importlib.util.module_from_spec(_spec)
sys.modules['L5'] = _L5
_spec.loader.exec_module(_L5)

print(f'Setup OK. Lesson module: {_L5.LESSON_NUMBER}')

Setup OK. Lesson module: M5.L5


In [2]:
# Run the 12 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L5, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 34 tests...

  PASS  test_content_hash_differs_on_different_text
  PASS  test_content_hash_distinguishes_punctuation
  PASS  test_content_hash_handles_empty_string
  PASS  test_content_hash_handles_unicode
  PASS  test_content_hash_is_16_chars
  PASS  test_content_hash_is_stable
  PASS  test_dead_letter_failed_at_is_isoformat
  PASS  test_dead_letter_preserves_attempt_count
  PASS  test_ingest_dedupe_by_content_not_source
  PASS  test_ingest_dlq_for_bad_docs
  PASS  test_ingest_dlq_for_unknown_content_type
  PASS  test_ingest_idempotent
  PASS  test_ingest_latency_ms_returns_stats
  PASS  test_ingest_latency_ms_within_budget
  PASS  test_ingest_math_invariants
  PASS  test_ingest_mixed_success_and_failure
  PASS  test_ingest_preserves_ingestion_order_in_result
  PASS  test_ingest_throughput_per_second_scales_with_batch
  PASS  test_ingested_doc_byte_count_matches_raw
  PASS  test_ingested_doc_doc_id_equals_content_hash
  PASS  test_parse_html_collapses_whitespace
  PASS  test_p

In [3]:
# === Section 1: parser registry ===
print('Parser registry:\n')
for name in _L5.PARSERS:
    print(f'  {name:<8} -> {_L5.PARSERS[name].__name__}')

Parser registry:

  txt      -> parse_txt
  md       -> parse_md
  html     -> parse_html
  json     -> parse_json


In [4]:
# === Section 2: content hash (idempotency key) ===
h1 = _L5.content_hash('hello world')
h2 = _L5.content_hash('hello world')
h3 = _L5.content_hash('hello WORLD')
print(f'content_hash("hello world") = {h1}')
print(f'content_hash("hello world") = {h2}  (same -> idempotent)')
print(f'content_hash("hello WORLD") = {h3}  (diff -> different)')

content_hash("hello world") = b94d27b9934d3e08
content_hash("hello world") = b94d27b9934d3e08  (same -> idempotent)
content_hash("hello WORLD") = a53bd0a9374bb36b  (diff -> different)


In [5]:
# === Section 3: single-doc parse ===
md_doc = _L5.SAMPLE_INPUTS[1]
parsed = _L5.parse_one(md_doc)
print(f'source: {parsed.source}')
print(f'parser: {parsed.parser}')
print(f'text:   {parsed.text!r}')
print(f'hash:   {parsed.content_hash}')

source: hr/401k.md
parser: md
text:   '401k Match\n\n100% of the first 3% of salary.'
hash:   144211693d415227


In [6]:
# === Section 4: idempotency (re-ingest the same doc) ===
svc = _L5.IngestionService()
r1 = svc.ingest([_L5.SAMPLE_INPUTS[0]])
print(f'first ingest:  ingested={len(r1.ingested)}, dupes={len(r1.duplicates)}, dlq={len(r1.dead_letter)}')
r2 = svc.ingest([_L5.SAMPLE_INPUTS[0]])
print(f'second ingest: ingested={len(r2.ingested)}, dupes={len(r2.duplicates)}, dlq={len(r2.dead_letter)}')

first ingest:  ingested=1, dupes=0, dlq=0
second ingest: ingested=0, dupes=1, dlq=0


In [7]:
# === Section 5: DLQ behavior ===
svc2 = _L5.IngestionService()
bad = [_L5.Document(source='bad.json', raw='{"text": missing}', content_type='json')]
r3 = svc2.ingest(bad)
print(f'bad json -> dlq: {len(r3.dead_letter)}')
for dl in r3.dead_letter:
    print(f'  {dl.source}: {dl.error}')

bad json -> dlq: 1
  bad.json: ValueError: invalid JSON: Expecting value: line 1 column 10 (char 9)


In [8]:
# === Section 6: end-to-end ingest of 7 mixed docs ===
svc3 = _L5.IngestionService()
r = svc3.ingest(_L5.SAMPLE_INPUTS)
print(f'ingested:    {len(r.ingested)}')
for d in r.ingested:
    print(f'  {d.doc_id}  {d.parser:<6}  {d.char_count:>4} chars  {d.source}')
print(f'\nduplicates:  {len(r.duplicates)}')
for src, h in r.duplicates:
    print(f'  {src}  ->  existing hash {h}')
print(f'\ndead_letter: {len(r.dead_letter)}')
for dl in r.dead_letter:
    print(f'  {dl.source}: {dl.error}')

ingested:    4
  90928a1e93ec355a  txt       34 chars  hr/policy.txt
  144211693d415227  md        43 chars  hr/401k.md
  04a7b549f7d09f69  html      37 chars  web/expense.html
  9b6d6573eb66f19e  json      49 chars  api/qbr.json

duplicates:  1
  hr/policy-copy.txt  ->  existing hash 90928a1e93ec355a

dead_letter: 2
  api/bad.json: ValueError: invalid JSON: Expecting value: line 1 column 10 (char 9)
  weird.xyz: ValueError: no parser for content_type='xyz'


In [9]:
# === Section 7: idempotency over a second full run ===
before = len(svc3.store)
r2 = svc3.ingest(_L5.SAMPLE_INPUTS)
after = len(svc3.store)
print(f'store size before re-ingest: {before}')
print(f'store size after  re-ingest: {after}  (should be equal)')
print(f'duplicates detected:         {len(r2.duplicates)}')
print(f'new docs ingested:           {len(r2.ingested)}  (should be 0)')

store size before re-ingest: 4
store size after  re-ingest: 4  (should be equal)
duplicates detected:         5
new docs ingested:           0  (should be 0)


## Architecture (Mermaid)

```mermaid
flowchart LR
    A[raw doc: txt/md/html/json] --> B{parser registry}
    B -->|txt| P1[parse_txt]
    B -->|md| P2[parse_md: drop front-matter]
    B -->|html| P3[parse_html: drop script/style]
    B -->|json| P4[parse_json: pull 'text' field]
    P1 --> N[normalize: strip, lowercase, dedup whitespace]
    P2 --> N
    P3 --> N
    P4 --> N
    N --> H[content_hash: SHA-256 first 16 chars]
    H --> D{hash in store?}
    D -->|yes| X[duplicate: skip, log]
    D -->|no| S[store: hash -> text + metadata]
    S --> OK[done]
    B -->|parse error| DLQ[dead-letter queue]
```

**Read as:** Parse, hash, dedupe, store. Anything that fails parsing goes to the DLQ — never the store. Same hash twice = same doc; the second one is a duplicate.

## Reflect (10 min)

1. Why is content-hash dedupe better than path-based dedupe?
2. What happens if a doc is edited and re-ingested? How do you handle stale entries?
3. Why must the DLQ be per-document, not per-batch?
4. The order parse → hash → dedupe → store matters. What's the failure mode if you reorder?
5. (Stretch) Add a `metrics` dict to `IngestionService.ingest` that returns: `n_in, n_ok, n_dup, n_dlq, total_bytes, total_chars`.

---

**Read next:**
- `lessons/m06-chunking/L6-chunking.ipynb` — fixed-window, recursive, structure-aware chunkers
- `lessons/m07-retrieval/L7-retrieval-ranking.ipynb` — BM25, dense, RRF, MMR
- `SYLLABUS.md` — the 18 modules and 21-day plan